# 03c — LSTM Autoencoder (Farm A)

**Global** sequence autoencoder on **`wind-farm-a-features-forecasting`** (same Delta table as **02** / **03a** / **03b** when that pipeline is used): gap-aware **24 h** windows (**144** ten-minute steps), train only on windows where **every** timestep is **`if_fit_eligible`**, score **reconstruction MSE** per row (last timestep of each window). **`lstm_recon_error_norm`** ∈ [0, 1] is min–max scaled using **only `if_fit_eligible` rows** (higher = more anomalous) for **Stage 4** ensemble joins with **`if_anomaly_score_norm`** (03a) and **`xgb_risk_score_norm`** (03b) on **`asset_id`**, **`time_stamp`**, **`id`**.

**Dependencies** — Run **`02_feature_engineering`** through CELL 8 so **`if_fit_eligible`** and **`in_anomaly_window`** exist. **`TRAIN_END` in CELL 1** must match **02** and **03a**.

**Outputs** — Delta **`wind-farm-a-lstm-scores`**, **`wind-farm-a-lstm-feature-recon-error`**; MLflow experiment **`DSMLC-Final-Comp-2026-lstm-autoencoder`**.

**Memory** — Full-table `toPandas()` can OOM on large farms; reduce scope or sample if needed.


In [0]:
%pip install mlflow tensorflow


In [0]:
dbutils.library.restartPython()


In [0]:
# CELL 1 — Configuration (align TRAIN_END with 02_feature_engineering and 03a)

import warnings

import mlflow
import numpy as np
import pandas as pd
import tensorflow as tf
from pyspark.sql import functions as F
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
try:
    tf.keras.utils.disable_interactive_logging()
except AttributeError:
    pass

CATALOG = "wind-turbine-silver"
FEATURE_TABLE = "wind-farm-a-features-forecasting"
OUTPUT_TABLE_SCORES = "wind-farm-a-lstm-scores"
OUTPUT_TABLE_FEATURE_RECON = "wind-farm-a-lstm-feature-recon-error"
EVENT_TABLE = "wind-farm-a-event-info"

TRAIN_END = "2023-01-01"

SEQ_LENGTH = 144
MAX_GAP_MINUTES = 15
LATENT_DIM = 32
BATCH_SIZE = 32
MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 3

WEIGHTS_PATH = "/tmp/lstm_autoencoder_farm_a.weights.h5"
FEATURE_RECON_CSV = "/tmp/lstm_feature_recon_error.csv"

mlflow.set_experiment(
    "/Users/"
    + spark.sql("SELECT current_user()").first()[0]
    + "/DSMLC-Final-Comp-2026-lstm-autoencoder"
)

# Match 03a / 03b: no labels, targets, masks, or metadata in model inputs
IF_EXCLUDE_STATUS_HISTORY = False
STATUS_HISTORY_COLS = [
    "downtime_frac_24h",
    "downtime_frac_7d",
    "downtime_frac_30d",
    "derated_frac_24h",
    "derated_frac_7d",
    "derated_frac_30d",
    "status_change_count_7d",
    "status_change_count_30d",
    "hours_since_last_downtime",
]

EXCLUDE_COLS = {
    "asset_id",
    "time_stamp",
    "id",
    "train_test",
    "status_type_id",
    "event_id",
    "farm",
    "event_label",
    "event_description",
    "risk_score",
    "hours_to_next_anomaly",
    "hours_to_anomaly_linked_downtime",
    "risk_score_anomaly_linked_downtime",
    "hours_to_next_operator_warning",
    "risk_score_operator_warning",
    "next_anomaly_event_id",
    "hours_to_failure_onset",
    "risk_score_onset",
    "next_failure_onset_ts",
    "next_onset_event_id",
    "next_onset_event_overlaps",
    "in_overlap_event_period",
    "is_usable_supervised_next",
    "if_fit_eligible",
    "in_anomaly_window",
}
if IF_EXCLUDE_STATUS_HISTORY:
    EXCLUDE_COLS |= set(STATUS_HISTORY_COLS)

print(
    "LSTM inputs — status history:",
    "excluded" if IF_EXCLUDE_STATUS_HISTORY else "included (default)",
)


In [0]:
# CELL 2 — Load and validate

df_spark = spark.table(f"`{CATALOG}`.`{FEATURE_TABLE}`")

required = [
    "asset_id",
    "time_stamp",
    "id",
    "train_test",
    "status_type_id",
    "if_fit_eligible",
    "in_anomaly_window",
]
missing = [c for c in required if c not in df_spark.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing} — run 02_feature_engineering through CELL 8 first"
    )

feature_cols = [c for c in df_spark.columns if c not in EXCLUDE_COLS]
if len(feature_cols) == 0:
    raise ValueError("No feature columns after EXCLUDE_COLS — run 02_feature_engineering first")

print(f"Feature count (LSTM inputs): {len(feature_cols)}")
print(f"Turbine count: {df_spark.select('asset_id').distinct().count():,}")
print(
    "Rows if_fit_eligible:",
    df_spark.filter(F.col("if_fit_eligible")).count(),
)
print(
    "Rows in_anomaly_window:",
    df_spark.filter(F.col("in_anomaly_window")).count(),
)


In [0]:
# CELL 2a — Gap detection and sequence construction
# OOM risk: full Farm A toPandas(); use sampling in dev if needed.

meta_cols = [
    "asset_id",
    "time_stamp",
    "id",
    "train_test",
    "status_type_id",
    "if_fit_eligible",
    "in_anomaly_window",
]
select_cols = meta_cols + feature_cols
pdf = df_spark.select(*select_cols).orderBy("asset_id", "time_stamp").toPandas()
pdf["time_stamp"] = pd.to_datetime(pdf["time_stamp"])
pdf["if_fit_eligible"] = pdf["if_fit_eligible"].astype(bool)

for c in feature_cols:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
pdf[feature_cols] = pdf[feature_cols].fillna(0.0)

train_entries = []
score_windows = []
stats = {
    "n_train_windows": 0,
    "n_score_windows": 0,
    "n_segments_too_short": 0,
    "n_train_skipped_ineligible": 0,
    "per_turbine": {},
}

for asset_id, g in pdf.groupby("asset_id", sort=False):
    g = g.sort_values("time_stamp").reset_index(drop=True)
    n = len(g)
    boundaries = [0]
    for i in range(1, n):
        dt_min = (g.loc[i, "time_stamp"] - g.loc[i - 1, "time_stamp"]).total_seconds() / 60.0
        if dt_min > MAX_GAP_MINUTES:
            boundaries.append(i)
    boundaries.append(n)

    pt = {"train": 0, "score": 0, "skipped_ineligible": 0, "short_seg": 0}

    for bi in range(len(boundaries) - 1):
        b, e = boundaries[bi], boundaries[bi + 1]
        seg = g.iloc[b:e].reset_index(drop=True)
        slen = len(seg)
        if slen < SEQ_LENGTH:
            stats["n_segments_too_short"] += 1
            pt["short_seg"] += 1
            continue
        Xs = seg[feature_cols].to_numpy(dtype=np.float32)
        elig = seg["if_fit_eligible"].to_numpy()

        for j in range(slen - SEQ_LENGTH + 1):
            win = Xs[j : j + SEQ_LENGTH]
            last = seg.iloc[j + SEQ_LENGTH - 1]
            ts_last = last["time_stamp"]

            score_windows.append(win)
            stats["n_score_windows"] += 1
            pt["score"] += 1

            if elig[j : j + SEQ_LENGTH].all():
                train_entries.append((win.copy(), ts_last, str(asset_id)))
                stats["n_train_windows"] += 1
                pt["train"] += 1
            else:
                stats["n_train_skipped_ineligible"] += 1
                pt["skipped_ineligible"] += 1

    stats["per_turbine"][str(asset_id)] = pt

train_entries.sort(key=lambda x: x[1])
X_train_list = [t[0] for t in train_entries]
if len(X_train_list) == 0:
    raise ValueError("No eligible training windows — check if_fit_eligible / SEQ_LENGTH / gaps")

X_train = np.stack(X_train_list, axis=0)
X_score = np.stack(score_windows, axis=0)

score_row_index = []
wi = 0
for asset_id, g in pdf.groupby("asset_id", sort=False):
    g = g.sort_values("time_stamp").reset_index(drop=True)
    n = len(g)
    boundaries = [0]
    for i in range(1, n):
        dt_min = (g.loc[i, "time_stamp"] - g.loc[i - 1, "time_stamp"]).total_seconds() / 60.0
        if dt_min > MAX_GAP_MINUTES:
            boundaries.append(i)
    boundaries.append(n)
    for bi in range(len(boundaries) - 1):
        b, e = boundaries[bi], boundaries[bi + 1]
        seg = g.iloc[b:e].reset_index(drop=True)
        slen = len(seg)
        if slen < SEQ_LENGTH:
            continue
        for j in range(slen - SEQ_LENGTH + 1):
            last = seg.iloc[j + SEQ_LENGTH - 1]
            score_row_index.append(
                {
                    "asset_id": last["asset_id"],
                    "time_stamp": last["time_stamp"],
                    "id": last["id"],
                    "train_test": last["train_test"],
                    "status_type_id": last["status_type_id"],
                    "if_fit_eligible": last["if_fit_eligible"],
                    "in_anomaly_window": last["in_anomaly_window"],
                    "_win_idx": wi,
                }
            )
            wi += 1

assert wi == len(score_windows) == X_score.shape[0], "score index mismatch"
score_meta_df = pd.DataFrame(score_row_index)

print(f"Training windows (all eligible): {stats['n_train_windows']:,}")
print(f"Scoring windows: {stats['n_score_windows']:,}")
print(f"Segments shorter than SEQ_LENGTH: {stats['n_segments_too_short']:,}")
print(
    "Windows skipped for training (any ineligible in window):",
    f"{stats['n_train_skipped_ineligible']:,}",
)
print(f"X_train shape: {X_train.shape}, X_score shape: {X_score.shape}")
print("Per-turbine sample (first 5 keys):", list(stats["per_turbine"].keys())[:5])
for k in list(stats["per_turbine"].keys())[:5]:
    print(k, stats["per_turbine"][k])


In [0]:
# CELL 3 — Model architecture

n_features = len(feature_cols)

inp = layers.Input(shape=(SEQ_LENGTH, n_features))
x = layers.LSTM(64, return_sequences=True)(inp)
latent = layers.LSTM(LATENT_DIM, return_sequences=False)(x)
r = layers.RepeatVector(SEQ_LENGTH)(latent)
x = layers.LSTM(32, return_sequences=True)(r)
x = layers.LSTM(64, return_sequences=True)(x)
out = layers.TimeDistributed(layers.Dense(n_features))(x)

model = keras.Model(inputs=inp, outputs=out, name="lstm_autoencoder_farm_a")
model.compile(optimizer="adam", loss="mse")
model.summary()


In [0]:
# CELL 4 — Training (temporal 80/20 split, MLflow, weights artifact)

n = X_train.shape[0]
if n < 2:
    raise ValueError("Need at least 2 training windows for train/val split")
split = int(n * 0.8)
split = max(1, min(split, n - 1))
X_tr, X_va = X_train[:split], X_train[split:]

cb = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
    verbose=1,
)

with mlflow.start_run(run_name="LSTM_Farm_A_parent"):
    mlflow.log_param("SEQ_LENGTH", SEQ_LENGTH)
    mlflow.log_param("MAX_GAP_MINUTES", MAX_GAP_MINUTES)
    mlflow.log_param("LATENT_DIM", LATENT_DIM)
    mlflow.log_param("BATCH_SIZE", BATCH_SIZE)
    mlflow.log_param("MAX_EPOCHS", MAX_EPOCHS)
    mlflow.log_param("n_features", n_features)
    mlflow.log_param("n_train_windows", n)
    mlflow.log_param("n_val_windows", X_va.shape[0])

    history = model.fit(
        X_tr,
        X_tr,
        validation_data=(X_va, X_va),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[cb],
        verbose=1,
    )

    for epoch, (loss, vloss) in enumerate(
        zip(history.history["loss"], history.history["val_loss"]), start=1
    ):
        mlflow.log_metric("train_loss", float(loss), step=epoch)
        mlflow.log_metric("val_loss", float(vloss), step=epoch)

    final_train = float(history.history["loss"][-1])
    final_val = float(history.history["val_loss"][-1])
    mlflow.log_metric("final_train_loss", final_train)
    mlflow.log_metric("final_val_loss", final_val)

    model.save_weights(WEIGHTS_PATH)
    mlflow.log_artifact(WEIGHTS_PATH)

    stopped_epoch = len(history.history["loss"])
    es = getattr(cb, "stopped_epoch", 0) or 0
    mlflow.log_param("epochs_ran", stopped_epoch)
    mlflow.log_param("early_stopping_triggered", es > 0)

print(f"Final train loss: {final_train:.6f}")
print(f"Final val loss:   {final_val:.6f}")
print(f"Epochs run: {stopped_epoch}, early_stopping triggered: {es > 0}, stopped_epoch attr: {es}")


In [0]:
# CELL 5 — Reconstruction error scoring + normalization

pred_score = model.predict(X_score, batch_size=BATCH_SIZE, verbose=1)
mse_win = np.mean((X_score - pred_score) ** 2, axis=(1, 2))

score_meta_df = score_meta_df.copy()
score_meta_df["lstm_recon_error"] = mse_win

key_cols = ["asset_id", "id"]
# Step-1 windows: each (asset_id, id) is the last row of at most one scoring window — .max() is redundant but harmless if logic changes.
agg_err = score_meta_df.groupby(key_cols, as_index=False)["lstm_recon_error"].max()
agg_err = agg_err.rename(columns={"lstm_recon_error": "_err"})

df_scored = pdf[meta_cols].copy()
df_scored["lstm_recon_error"] = 0.0
df_scored = df_scored.merge(agg_err, on=key_cols, how="left")
df_scored["lstm_recon_error"] = df_scored["_err"].fillna(0.0)
df_scored = df_scored.drop(columns=["_err"])

elig_err = df_scored.loc[df_scored["if_fit_eligible"], "lstm_recon_error"]
e_min = float(elig_err.min())
e_max = float(elig_err.max())
if e_max <= e_min:
    df_scored["lstm_recon_error_norm"] = 0.5
else:
    df_scored["lstm_recon_error_norm"] = (
        (df_scored["lstm_recon_error"] - e_min) / (e_max - e_min)
    ).clip(0, 1)

print(
    f"Norm fit on if_fit_eligible rows: min={e_min:.6e}, max={e_max:.6e}, n={elig_err.shape[0]:,}"
)


In [0]:
# CELL 6 — Per-feature reconstruction error (global diagnostic)

MAX_WINDOWS_FEATURE_RECON = min(5000, X_score.shape[0])
X_sub = X_score[:MAX_WINDOWS_FEATURE_RECON]
pred_sub = model.predict(X_sub, batch_size=BATCH_SIZE, verbose=0)
diff2 = (X_sub - pred_sub) ** 2
mean_per_f = diff2.mean(axis=(0, 1))
feat_df = pd.DataFrame(
    {"feature": feature_cols, "mean_recon_mse": mean_per_f.astype(float)}
).sort_values("mean_recon_mse", ascending=False)

feat_df.to_csv(FEATURE_RECON_CSV, index=False)

runs = mlflow.search_runs(
    filter_string="tags.mlflow.runName = 'LSTM_Farm_A_parent'",
    max_results=1,
)
parent_run_id = runs.iloc[0].run_id
with mlflow.start_run(run_id=parent_run_id):
    mlflow.log_artifact(FEATURE_RECON_CSV)

spark.createDataFrame(feat_df).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"`{CATALOG}`.`{OUTPUT_TABLE_FEATURE_RECON}`")
print(f"Saved per-feature recon -> `{CATALOG}`.`{OUTPUT_TABLE_FEATURE_RECON}`")
print(feat_df.head(15).to_string(index=False))


In [0]:
# CELL 7 — Validation against event labels

events = spark.table(f"`{CATALOG}`.`{EVENT_TABLE}`").toPandas()
if "asset" in events.columns and "asset_id" not in events.columns:
    events = events.rename(columns={"asset": "asset_id"})

df_eval = df_scored.merge(
    events[
        [
            "asset_id",
            "event_id",
            "event_label",
            "event_start_id",
            "event_end_id",
            "event_description",
        ]
    ],
    on="asset_id",
    how="inner",
)
df_eval["in_event"] = (df_eval["id"] >= df_eval["event_start_id"]) & (
    df_eval["id"] <= df_eval["event_end_id"]
)
df_eval = df_eval[df_eval["in_event"]].copy()

print(f"Rows inside event windows: {len(df_eval):,}\n")
print("Mean lstm_recon_error_norm by event_label:")
by_label = df_eval.groupby("event_label")["lstm_recon_error_norm"].agg(["mean", "std", "count"])
print(by_label)

anomaly_rows = df_eval[df_eval["event_label"] == "anomaly"]
if not anomaly_rows.empty:
    print("\nMean lstm_recon_error_norm by fault type (anomaly rows):")
    by_fault = (
        anomaly_rows.groupby("event_description")["lstm_recon_error_norm"]
        .mean()
        .sort_values(ascending=False)
    )
    print(by_fault)

with mlflow.start_run(run_id=parent_run_id):
    for label, row in by_label.iterrows():
        mlflow.log_metric(f"lstm_norm_mean_{label}", float(row["mean"]))
        mlflow.log_metric(f"lstm_norm_count_{label}", float(row["count"]))
    if not anomaly_rows.empty:
        for fi, (desc, v) in enumerate(by_fault.items()):
            mlflow.log_metric(f"lstm_norm_fault_{fi}", float(v))
            mlflow.log_param(f"lstm_norm_fault_{fi}_desc", str(desc)[:200])


In [0]:
# CELL 8 — Output schema validation and Delta save

required_out = [
    "asset_id",
    "time_stamp",
    "id",
    "train_test",
    "status_type_id",
    "lstm_recon_error",
    "lstm_recon_error_norm",
    "if_fit_eligible",
    "in_anomaly_window",
]
for col in required_out:
    if col not in df_scored.columns:
        raise ValueError(f"Output missing required column: {col}")

print("Schema:")
print(df_scored.dtypes)
print("\nNull counts:")
print(df_scored.isnull().sum())

spark.createDataFrame(df_scored).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"`{CATALOG}`.`{OUTPUT_TABLE_SCORES}`")

saved = spark.table(f"`{CATALOG}`.`{OUTPUT_TABLE_SCORES}`")
print(f"\nSaved rows: {saved.count():,}")
print("lstm_recon_error_norm describe:")
print(df_scored["lstm_recon_error_norm"].describe())
